In [ ]:
# Interactive visualization of IHK microdata points
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Convert to WGS84 for web mapping and fill NaN employment values
IHK_gdf_wgs84 = IHK_gdf.to_crs('EPSG:4326').copy()
IHK_gdf_wgs84['empl'] = IHK_gdf_wgs84['empl'].fillna(1)  # Fill NaN with 1 for size mapping

# === OPTION 1: Interactive Plotly scatter_map ===
# Shows individual points, hover reveals details, supports zooming
fig = px.scatter_map(
    IHK_gdf_wgs84,
    lat=IHK_gdf_wgs84.geometry.y,
    lon=IHK_gdf_wgs84.geometry.x,
    color='empl',
    size='empl',
    hover_name='ihk_branch_desc',
    hover_data={'empl': ':.0f', 'nace_desc': True},
    color_continuous_scale='YlOrRd',
    size_max=25,
    zoom=11,
    center={'lat': IHK_gdf_wgs84.geometry.y.mean(), 'lon': IHK_gdf_wgs84.geometry.x.mean()},
    title='IHK Berlin Business Microdata - Employment Size',
    map_style='open-street-map'
)
fig.update_layout(height=800, width=1200)

# Save to HTML for interactive viewing
output_path = REPORT_ROOT / 'ihk_microdata_interactive.html'
fig.write_html(str(output_path))
print(f"Interactive map saved to: {output_path}")

In [ ]:
# === OPTION 3: Transparency + Jitter (reveals overlapping points) ===
# For static visualization: add small random spatial noise and use transparency
import matplotlib.pyplot as plt
from shapely.affinity import translate as translate_geom

# Create jittered copy using vectorized offset approach
np.random.seed(42)
jitter_scale = 20
jitter_x = np.random.normal(0, jitter_scale, len(IHK_gdf))
jitter_y = np.random.normal(0, jitter_scale, len(IHK_gdf))

# Apply jitter to geometries
IHK_jittered = IHK_gdf.copy()
IHK_jittered['geometry'] = [translate_geom(geom, xoff=dx, yoff=dy) 
                             for geom, dx, dy in zip(IHK_jittered.geometry, jitter_x, jitter_y)]

fig, ax = plt.subplots(figsize=(16, 14))
berlin.plot(ax=ax, alpha=0.2, edgecolor='black', facecolor='none', linewidth=2)
grid.plot(ax=ax, column='empl', cmap='YlGn', alpha=0.3, legend=False, edgecolor='none')

# Main visualization: size and color by employment
scatter = ax.scatter(
    IHK_jittered.geometry.x,
    IHK_jittered.geometry.y,
    s=IHK_jittered['empl'].fillna(1).clip(upper=500),  # Cap size for outliers
    c=IHK_jittered['empl'],
    cmap='YlOrRd',
    alpha=0.5,  # Transparency reveals overlaps
    edgecolors='darkred',
    linewidth=0.5
)
cbar = plt.colorbar(scatter, ax=ax, label='Number of Employees')
ax.set_title('IHK Berlin Microdata: Individual Business Points with Jitter', fontsize=14)
ax.set_xlabel('Easting (EPSG:3035)')
ax.set_ylabel('Northing (EPSG:3035)')
plt.tight_layout()
plt.show()

In [ ]:
# === OPTION 4: Contour density + individual points ===
# Show both density patterns and individual locations
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(16, 14))

# Background: density heatmap
points = np.column_stack([IHK_gdf.geometry.x, IHK_gdf.geometry.y])
if len(points) > 1:
    kde = gaussian_kde(points.T, weights=IHK_gdf['empl'].fillna(1))
    
    # Create grid for KDE evaluation
    x_min, x_max = IHK_gdf.geometry.x.min(), IHK_gdf.geometry.x.max()
    y_min, y_max = IHK_gdf.geometry.y.min(), IHK_gdf.geometry.y.max()
    xx, yy = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]
    positions = np.vstack([xx.ravel(), yy.ravel()])
    z = kde(positions).reshape(xx.shape)
    
    # Plot contours of density
    contourf = ax.contourf(xx, yy, z, levels=15, cmap='Blues', alpha=0.4)
    contour = ax.contour(xx, yy, z, levels=5, colors='steelblue', alpha=0.3, linewidths=0.5)
    plt.colorbar(contourf, ax=ax, label='Business Density (weighted)')

# Overlay individual points
scatter = ax.scatter(
    IHK_gdf.geometry.x,
    IHK_gdf.geometry.y,
    s=np.sqrt(IHK_gdf['empl'].fillna(1)) * 3,
    c=IHK_gdf['empl'],
    cmap='YlOrRd',
    alpha=0.6,
    edgecolors='darkred',
    linewidth=0.5,
    zorder=5
)
plt.colorbar(scatter, ax=ax, label='Employees per Business')
ax.set_title('IHK Berlin Microdata: Density + Individual Points', fontsize=14)
ax.set_xlabel('Easting (EPSG:3035)')
ax.set_ylabel('Northing (EPSG:3035)')
plt.tight_layout()
plt.show()